# 360도 사진 1장 → 3D (Google Colab)

등장방형(equirectangular, 가로:세로 2:1) 360 사진 한 장에서 AI 깊이 추정으로 3D 포인트를 만들어,
가우시안 스플래팅(.ply) 형식으로 저장합니다. 결과는 **https://gojosuperman.github.io/hotel/viewer.html** 에 드래그해서 봅니다.

- 사용 모델: Depth Anything V2 Small (Apache-2.0 — 한국 사용/상업 제약 없음)
- 무료 T4로 충분, 전체 몇 분이면 완료 (GPU 없이도 동작하나 느림)
- 한계: 사진에 찍힌 표면만 3D가 됩니다. 시점을 크게 옮기면 가려졌던 곳이 구멍으로 보입니다.
  실내(객실) 사진일수록 효과가 좋고, 야외 원경은 배경막처럼 평평하게 남습니다.

**준비**: ① 런타임 유형 → T4 GPU ② 드라이브 최상위에 `pano_input` 폴더를 만들고 360 사진 1장(JPG/PNG) 업로드

In [ ]:
# 1) 드라이브 연결 + 360 사진 확인
from google.colab import drive
drive.mount('/content/drive')

import glob, os
files = sorted(glob.glob('/content/drive/MyDrive/pano_input/*.[jJpP][pPnN]*[gG]'))
assert files, 'pano_input 폴더가 없거나 JPG/PNG 사진이 없습니다.'
PANO_PATH = files[0]  # 여러 장이면 첫 번째 사용

from PIL import Image
Image.MAX_IMAGE_PIXELS = None
img = Image.open(PANO_PATH).convert('RGB')
w, h = img.size
print(f'사진: {os.path.basename(PANO_PATH)} ({w}x{h}, 비율 {w/h:.2f}:1)')
if abs(w / h - 2.0) > 0.1:
    print('⚠️ 경고: 표준 360 사진은 비율이 2:1입니다. 이 사진은 다르므로 결과가 왜곡될 수 있습니다.')

In [ ]:
# 2) AI 깊이 추정 (Depth Anything V2 — 처음 실행 시 모델 다운로드 ~100MB)
import torch, numpy as np
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
pipe = pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf', device=device)

W, H = 1600, 800   # 처리 해상도 (포인트 약 128만 개). 메모리 부족 시 1200, 600으로
img_r = img.resize((W, H), Image.LANCZOS)

disp = np.array(pipe(img_r)['depth'], dtype=np.float32)
disp = (disp - disp.min()) / (disp.max() - disp.min() + 1e-6)   # 0(멀다)~1(가깝다)로 정규화

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].imshow(img_r); ax[0].set_title('원본'); ax[0].axis('off')
ax[1].imshow(disp, cmap='inferno'); ax[1].set_title('추정 깊이 (밝을수록 가까움)'); ax[1].axis('off')
plt.show()

In [ ]:
# 3) 구면 역투영 → 가우시안 스플래팅 .ply 생성 → 드라이브 저장
import numpy as np, os

R_NEAR, R_FAR = 1.0, 40.0   # 가장 가까운/먼 표면까지의 거리(m 느낌). 방 사진이면 R_FAR=8 정도가 자연스러움
radius = R_NEAR + (R_FAR - R_NEAR) * (1.0 - disp) ** 1.5

# 각 픽셀을 구면 방향으로 배치 (뷰어 기준 y축 아래 방향)
v, u = np.mgrid[0:H, 0:W].astype(np.float32)
lon = (u / W - 0.5) * 2 * np.pi
lat = (0.5 - v / H) * np.pi
dx = np.cos(lat) * np.sin(lon)
dy = -np.sin(lat)
dz = np.cos(lat) * np.cos(lon)
xyz = np.stack([dx, dy, dz], -1) * radius[..., None]

rgb = np.asarray(img_r, dtype=np.float32) / 255.0
n = H * W
xyz = xyz.reshape(n, 3); rgb = rgb.reshape(n, 3); rad = radius.reshape(n)

# 3DGS ply 필드 구성 (SH 0차: 색상만)
SH_C0 = 0.28209479177387814
scale = np.log(rad * (2 * np.pi / W) * 2.0)   # 포인트 간격에 맞춘 가우시안 크기
names = ['x','y','z','nx','ny','nz','f_dc_0','f_dc_1','f_dc_2','opacity','scale_0','scale_1','scale_2','rot_0','rot_1','rot_2','rot_3']
arr = np.zeros(n, dtype=[(k, '<f4') for k in names])
arr['x'], arr['y'], arr['z'] = xyz.T
for i, k in enumerate(['f_dc_0','f_dc_1','f_dc_2']):
    arr[k] = (rgb[:, i] - 0.5) / SH_C0
arr['opacity'] = 5.0                      # 시그모이드 후 불투명
for k in ['scale_0','scale_1','scale_2']:
    arr[k] = scale
arr['rot_0'] = 1.0                        # 회전 없음(단위 쿼터니언)

base = os.path.splitext(os.path.basename(PANO_PATH))[0]
out = f'/content/drive/MyDrive/pano3d_{base}.ply'
header = ('ply\nformat binary_little_endian 1.0\n'
          f'element vertex {n}\n'
          + ''.join(f'property float {k}\n' for k in names)
          + 'end_header\n')
with open(out, 'wb') as f:
    f.write(header.encode('ascii')); f.write(arr.tobytes())
print(f'저장 완료: {out}  (포인트 {n:,}개, {os.path.getsize(out)/1e6:.0f}MB)')

## 4) 결과 보기

1. https://gojosuperman.github.io/hotel/viewer.html 접속
2. 드라이브(또는 G: 드라이브)의 `pano3d_<사진이름>.ply`를 화면에 드래그 (또는 클릭해서 선택)
3. 휠로 축소하면서 장면 **안쪽**으로 들어가 둘러보세요 — 촬영 지점이 장면의 중심입니다

**조정 팁** (3번 셀 수정 후 재실행)
- 실내(객실) 사진: `R_FAR = 8` 정도로 낮추면 공간감이 자연스러움
- 위아래가 뒤집혀 보이면: 뷰어 문제이니 `viewer.html`의 `cameraUp`을 `[0, 1, 0]`으로
- 더 촘촘한 3D: `W, H = 2048, 1024` (메모리 여유 필요)